# ST-03 — Siretisation, phase 2 : top 5 par commune

Matching probabiliste sur les etablissements non valides en phase 1, avec
blocking **communal**, maille d'origine du projet. La phase 3 elargira au
departement.

Cote SIRENE, le nom retenu suit la regle du projet : la meilleure des trois
enseignes ou de la denomination usuelle, avec la denomination de l'unite legale
en repli quand aucune n'est renseignee.

Les candidats sont pris dans l'echantillon SIRENE filtre produit par ST-02bis,
restreint aux SIREN issus de la sirenisation. La sirenisation doit donc etre
terminee avant de lancer cette phase.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from src.siretisation import CONFIG_ST, COLS_ETAB_EXPORT
from src.phases import executer_phase, estimer_charge
from src import domaine_ape
from src.excel_export import export_topn_excel, LABELS
from src.display import afficher_tableau, afficher_synthese
from config.settings import (
    ST_PERIMETRE, SIRENE_ETAB_FILTRE, SIRENE_ETAB_FILTRE_PARTITIONNE,
    ST_VALIDES_P1, ST_PHASE2, ST_VALIDES_P2, RESULTS_ST_DIR, BUDGET_MEMOIRE_GO,
)

RESULTS_ST_DIR.mkdir(parents=True, exist_ok=True)
VALIDES = {"VALIDE_FORT", "VALIDE"}

sirets_valides_p1 = set(
    pd.read_parquet(ST_VALIDES_P1)["siret_norm_ege"].dropna().astype(str))
sirets_valides_p1.discard("")
print(f"SIRET valides en P1 : {len(sirets_valides_p1):,}")

df_perimetre = pd.read_parquet(ST_PERIMETRE)
print(f"EGE a traiter : {len(df_perimetre):,}")

SIRET valides en P1 : 57,749
EGE a traiter : 44,101


## 1. Estimation de la charge

In [2]:
tailles = (pd.read_parquet(SIRENE_ETAB_FILTRE, columns=["code_commune_norm_etab"])
             .groupby("code_commune_norm_etab").size())
estimation = estimer_charge(df_perimetre, CONFIG_ST, tailles,
                            niveau_matching="commune")
display(estimation.head(15))

Maille          : commune
Paires a scorer : 9,787,625
Blocs           : 7913


,nb_structures,nb_candidats,paires,lot_max
31555,354.0,1928.0,682512.0,21611
06088,317.0,1505.0,477085.0,27685
44109,195.0,1291.0,251745.0,32274
34172,196.0,1245.0,244020.0,33467
33063,178.0,1314.0,233892.0,31709
59350,205.0,1113.0,228165.0,37436
35238,209.0,1023.0,213807.0,40729
75115,168.0,1126.0,189168.0,37004
67482,149.0,1079.0,160771.0,38616
42218,137.0,1012.0,138644.0,41172


## 2. Matching

In [3]:
df_top5 = executer_phase(
    df_perimetre, CONFIG_ST,
    chemin_partitionne=SIRENE_ETAB_FILTRE_PARTITIONNE,
    chemin_complet=SIRENE_ETAB_FILTRE,
    n=5, approfondi=False,
    identifiants_deja_valides=sirets_valides_p1,
    colonnes_sirene=COLS_ETAB_EXPORT,
    budget_go=BUDGET_MEMOIRE_GO,
    niveau_matching="commune",
)

df_top5 = df_top5.rename(columns={"siret": "siret_ref",
                                  "nom_sirene_retenu": "nom_etab_retenu",
                                  "identifiant_coherent": "siret_coherent",
                                  "score_ajuste": "score_global_ajuste"})

rang1 = df_top5[df_top5["rang"] == 1]
print(f"\nLignes top 5 : {len(df_top5):,}")
print(f"SIRET coherent au rang 1 : {int(rang1['siret_coherent'].sum()):,}")
print()
print(rang1["statut_candidat"].value_counts().to_string())

1 structures sans departement exploitable


Phase 2 (commune):   0%|          | 0/104 [00:00<?, ?it/s]

/home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/src/phases.py:139: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  df = pd.concat(resultats + orphelins, ignore_index=True) if (resultats or orphelins) \



Lignes top 5 : 215,024
SIRET coherent au rang 1 : 9,299

statut_candidat
VALIDE           26531
DOUTEUX          10252
VALIDE_FORT       5314
REJETE            1780
SANS_CANDIDAT      224


## 3. Controle du domaine d'activite

Une validation par le score ne suffit pas. Un candidat peut avoir le bon nom et
la bonne adresse mais une activite etrangere au champ FINESS — une SCI portant
les murs d'un etablissement, par exemple.

Un candidat n'est retenu que si son code APE est identique a celui de la
structure, ou s'il releve du domaine sanitaire, social ou medico-social.
L'egalite exacte prime. Parmi les candidats valides par le score, on retient le
mieux classe qui satisfait cette condition.

Quand aucun candidat valide n'est acceptable, la structure ressort en
`NON_FIABLE_APE` : le score la validait, l'activite l'en empeche.

In [4]:
df_top5 = domaine_ape.appliquer_controle(
    df_top5, col_id="NB_EgeId",
    col_ape_candidat="activitePrincipaleEtablissement",
    col_ape_structure="ape_referentiel",
)

rang1 = df_top5[df_top5["rang"] == 1]

afficher_tableau(domaine_ape.rapport(df_top5, "NB_EgeId"),
                 "Effet du controle d'activite", max_lignes=12)
print()
print(rang1["statut_structure"].value_counts().to_string())

Indicateur,Nombre
Structures traitées,44101
Validées par le score au rang 1,31845
—,
Validées après contrôle d'activité,32165
dont candidat de rang 1 conservé,25985
dont candidat remplacé,6180
Bloquées — NON_FIABLE_APE,3072
—,
retenues sur APE identique,5927
retenues sur le domaine,26238



statut_structure
VALIDE            27090
DOUTEUX            6860
VALIDE_FORT        5075
NON_FIABLE_APE     3072
REJETE             1780
SANS_CANDIDAT       224


## 4. Apercu

In [5]:
afficher_tableau(
    df_top5[["NB_EgeId", "TX_NomEgeLong", "siret_ref", "denominationUniteLegale",
             "nom_etab_retenu", "ape_finess", "ape_candidat", "ape_acceptable",
             "score_global_ajuste", "statut_candidat", "candidat_retenu",
             "statut_structure", "rang"]],
    "Apercu top 5 phase 2", max_lignes=12)

NB_EgeId,TX_NomEgeLong,siret_ref,denominationUniteLegale,nom_etab_retenu,ape_finess,ape_candidat,ape_acceptable,score_global_ajuste,statut_candidat,candidat_retenu,statut_structure,rang
10,CH INTERCOMMUNAL AIN VAL DE SAONE - PONT DE VEYLE,20003004700019,CENTRE HOSPITALIER INTERCOMMUNAL AIN VAL DE SAONE,CENTRE HOSPITALIER INTERCOMMUNAL AIN VAL SAONE,8610Z,8610Z,True,68.510000,VALIDE,True,VALIDE,1
10,CH INTERCOMMUNAL AIN VAL DE SAONE - PONT DE VEYLE,52028072800012,UNION COMMERCIALE ET ARTISANALE DE PONT DE VEYLE,UNION COMMERCIALE ARTISANALE PONT VEYLE,8610Z,9499Z,True,44.910000,DOUTEUX,False,,2
10,CH INTERCOMMUNAL AIN VAL DE SAONE - PONT DE VEYLE,20003004700027,CENTRE HOSPITALIER INTERCOMMUNAL AIN VAL DE SAONE,EHPAD SITE PONT VEYLE,8610Z,8710A,True,41.090000,DOUTEUX,False,,3
10,CH INTERCOMMUNAL AIN VAL DE SAONE - PONT DE VEYLE,26010389000026,CENTRE COMMUNAL D'ACTION SOCIALE,CENTRE COMMUNAL ACTION SOCIALE,8610Z,8899B,True,40.390000,DOUTEUX,False,,4
10,CH INTERCOMMUNAL AIN VAL DE SAONE - PONT DE VEYLE,44141340800010,RUGBY CLUB VEYLE SAONE,RUGBY CLUB VEYLE SAONE,8610Z,9312Z,False,38.850000,REJETE,False,,5
32,SERVICE DE SOINS INFIRMIERS A DOMICILE DE PONT DE VEYLE,20003004700027,CENTRE HOSPITALIER INTERCOMMUNAL AIN VAL DE SAONE,EHPAD SITE PONT VEYLE,8690D,8710A,True,61.090000,VALIDE,False,VALIDE,1
32,SERVICE DE SOINS INFIRMIERS A DOMICILE DE PONT DE VEYLE,20003004700068,CENTRE HOSPITALIER INTERCOMMUNAL AIN VAL DE SAONE,SERVICE ACCUEIL ALZHEIMER,8690D,8710A,True,52.600000,VALIDE,False,,2
32,SERVICE DE SOINS INFIRMIERS A DOMICILE DE PONT DE VEYLE,20003004700043,CENTRE HOSPITALIER INTERCOMMUNAL AIN VAL DE SAONE,DOTATION NON AFFECTEE,8690D,6619A,False,50.430000,VALIDE,False,,3
32,SERVICE DE SOINS INFIRMIERS A DOMICILE DE PONT DE VEYLE,20003004700019,CENTRE HOSPITALIER INTERCOMMUNAL AIN VAL DE SAONE,CENTRE HOSPITALIER INTERCOMMUNAL AIN VAL SAONE,8690D,8610Z,True,49.850000,DOUTEUX,False,,4
32,SERVICE DE SOINS INFIRMIERS A DOMICILE DE PONT DE VEYLE,20003004700050,CENTRE HOSPITALIER INTERCOMMUNAL AIN VAL DE SAONE,SSIAD,8690D,8690D,True,48.920000,VALIDE,True,,5


In [6]:
remplaces = rang1[(rang1["rang_retenu"].notna()) & (rang1["rang_retenu"] != 1)]
print(f"{len(remplaces):,} structures ou le candidat de rang 1 a ete ecarte")
if len(remplaces):
    afficher_tableau(
        remplaces[["NB_EgeId", "TX_NomEgeLong", "ape_finess", "ape_candidat",
                   "rang_retenu", "motif_ape", "statut_structure"]],
        "Candidats remplaces par le controle d'activite", max_lignes=10)

6,180 structures ou le candidat de rang 1 a ete ecarte


NB_EgeId,TX_NomEgeLong,ape_finess,ape_candidat,rang_retenu,motif_ape,statut_structure
32,SERVICE DE SOINS INFIRMIERS A DOMICILE DE PONT DE VEYLE,8690D,8710A,5.000000,APE_IDENTIQUE,VALIDE
919,EHPAD DU CHAVS SITE PONT DE VEYLE THOISSEY MONTMERLE,8710A,8710A,2.000000,DOMAINE_FINESS,VALIDE
565,CATTG ADULTES MONTREVEL,,0161Z,2.000000,DOMAINE_FINESS,VALIDE
658,Service de soins SNCF - AMBERIEU,,8299Z,2.000000,DOMAINE_FINESS,VALIDE
79,CMP ENFANTS-ADO BOURG-EN-BRESSE,,6512Z,2.000000,DOMAINE_FINESS,VALIDE
402,CATTG ENFANTS,,8710B,3.000000,DOMAINE_FINESS,VALIDE
403,CMP PERSONNES AGEES BOURG-EN-BRESSE,,8690F,2.000000,DOMAINE_FINESS,VALIDE
412,RÉSIDENCE SOCIALE PAUL BARBEROT,,8520Z,3.000000,DOMAINE_FINESS,VALIDE
438,RÉSIDENCE SOCIALE LES MARRONNIERS,5590Z,5590Z,4.000000,DOMAINE_FINESS,VALIDE
632,EML VASELINA CLINIQUE CONVERT,8219Z,8610Z,3.000000,APE_IDENTIQUE,VALIDE


## 5. Export

In [7]:
COLS_EXPORT = [
    "ID_Ege", "NB_EgeId", "NB_PmSmsseId", "TX_NumFinessEge", "TX_Siret",
    "TX_CategorieEntiteGeographiqueExercice", "TX_NomEgeLong",
    "cdcommune_norm_ege", "adresse_complete_ege",
    "ape_finess",
    "siret_ref", "siren", "siret_coherent", "bonus_applique",
    "denominationUniteLegale", "enseigne1Etablissement",
    "denominationUsuelleEtablissement", "nom_etab_retenu",
    "adresse_complete_etab", "codeCommuneEtablissement",
    "categorieJuridiqueUniteLegale", "activitePrincipaleEtablissement",
    "dateCreationEtablissement", "etablissementSiege",
    "ape_candidat", "ape_identique", "ape_dans_domaine", "ape_acceptable",
    "candidat_retenu", "rang_retenu", "motif_ape",
    "score_nom", "score_adresse", "score_global", "score_global_ajuste",
    "statut_candidat", "statut_structure", "rang",
]

df_top5["rang"] = df_top5["rang"].astype(int)
compteurs = export_topn_excel(df_top5, ST_PHASE2, COLS_EXPORT, sheet_name="Top5",
                              col_statut="statut_structure")

afficher_synthese({LABELS[s]: n for s, n in compteurs.items()},
                  "Synthese phase 2 siretisation (rang 1)")
print(f"\nFichier : {ST_PHASE2}")

Statut,Nb,% du total
Valide_fort,"5,075",11.5%
Valide,"27,090",61.4%
Douteux,"6,860",15.6%
Rejeté,"1,780",4.0%
Sans_candidat,224,0.5%
Non_fiable_APE,"3,072",7.0%
TOTAL,"44,101",100.0%



Fichier : /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/results/siretisation/siretisation_phase2_top5.xlsx


In [8]:
valides = rang1[rang1["statut_structure"].isin(VALIDES)][
    ["NB_EgeId", "siret_ref", "statut_structure"]].copy()
valides.to_parquet(ST_VALIDES_P2, index=False)
print(f"{len(valides):,} EGE valides en phase 2 -> {ST_VALIDES_P2}")

32,165 EGE valides en phase 2 -> /home/jovyan/work/finess+_refonte/projet_finess_sirene_finess+/data/processed/siretisation_valides_p2.parquet
